# FOXF1_bead — 10_array_center_distance_analysis

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 10 Array Center Distance Analysis


## Strategy

This notebook keeps the center-finding logic explicit and reviewable. It computes three center estimates from the fixed **round-1 DAPI core mask** for each position:

1. `round1_centroid`: centroid of the largest connected component of the round-1 mask.
2. `round1_dt_peak`: peak of the Euclidean distance transform inside the same mask.
3. `round1_hull_circle_fit`: least-squares circle-fit center on the convex-hull boundary of the round-1 mask.

A single default center is then chosen by an objective criterion: for each candidate center, we compute the coefficient of variation of distances from that center to the convex-hull boundary. The candidate with the lowest radial CV is selected. If the selected point falls outside the round-1 mask, it is snapped back to the nearest in-mask pixel.

This gives us multiple interpretable solutions while still forcing a single reproducible center for downstream center-distance traces.


In [ ]:

from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

MPLCONFIGDIR = ROOT / ".matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("ROOT:", ROOT)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from IPython.display import display
from scipy import ndimage as ndi
from skimage import measure, morphology

from scripts import pipeline_common as common
from scripts import live_fixed_quantification as lfq
from scripts import live_fixed_alignment as lfa


## Paths And Configuration

In [ ]:

POSITION_MANIFEST = ROOT / "results/manifests/analysis_position_manifest.tsv"
LIVE_FIXED_TSV = ROOT / "results/annotations/live_fixed_alignment_transforms.tsv"
FIXED_GLOBAL_BG_TSV = ROOT / "results/measurements/live_fixed_small/fixed_global_background_params.tsv"

OUT_DIR = ROOT / "results/measurements/live_fixed_small"
FIGURE_DIR_ROOT = ROOT / "results/figures"
FIG_DIR = FIGURE_DIR_ROOT / "10"
CANDIDATE_FIGURE_DIR = FIG_DIR / "candidate_figures"
ALTERNATE_FIGURE_DIR = FIG_DIR / "alternate_figures"
MASK_DIR = ROOT / "results/masks/live_fixed_small"
LIVE_MASK_FINAL_DIR = MASK_DIR / "live_mask_final04_npz"
LIVE_FIELD_NPZ = OUT_DIR / "live_yfp_flatfield_field.npz"
LIVE_FIELD_SUMMARY_TSV = OUT_DIR / "live_yfp_flatfield_summary.tsv"
LIVE_MASK_SUMMARY_TSV = OUT_DIR / "live_mask_transfer_summary.tsv"
PER_IMAGE_BG_TSV = OUT_DIR / "live_yfp_per_image_background_params.tsv"
for path in [OUT_DIR, FIG_DIR, CANDIDATE_FIGURE_DIR, ALTERNATE_FIGURE_DIR, MASK_DIR, LIVE_MASK_FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LIVE_COHORT_ID = "2026-01-22_day2_live"
FIXED_COHORT_ID = "2026-01-22_day2_fix"
LIVE_YFP_KEYWORDS = ["tagyfp", "foxf1", "yfp"]
CENTER_BIN_UM = 35.0
CENTER_REPRESENTATIVE_POSITIONS = ["1-1", "2-6", "6-6"]
CENTER_METHOD_ORDER = [
    "round1_centroid",
    "round1_dt_peak",
    "round1_hull_circle_fit",
]
CENTER_METHOD_COLORS = {
    "round1_centroid": "#ffffff",
    "round1_dt_peak": "#5ec962",
    "round1_hull_circle_fit": "#ff66c4",
    "selected": "#39d353",
}
TRACE_PLOT_YMIN = -0.5
TRACE_YLIM_SUPPORT_FRACTION = 2.0 / 3.0
TRACE_YLIM_UPPER_QUANTILE = 0.95
TRACE_YLIM_UPPER_PAD = 0.10
CENTER_TRACE_FIXEDWIDTH_XMIN = 0.0
CENTER_TRACE_EQUAL_SUPPORT_XMIN = 50.0
CENTER_TRACE_XMAX = 750.0
CENTER_TRACE_XLABEL = "Distance from anterior (um)"

MEASUREMENT_ORDER = [
    "fixed_tagyfp_bgz_over_dapi_gate",
    "fixed_sox2_bgz_over_dapi_gate",
    "fixed_t_bgz_over_dapi_gate",
]
MEASUREMENT_LABELS = {
    "fixed_tagyfp_bgz_over_dapi_gate": "FOXF1-YFP reporter (fixed) / DAPI",
    "fixed_sox2_bgz_over_dapi_gate": "SOX2 / DAPI",
    "fixed_t_bgz_over_dapi_gate": "TBXT / DAPI",
}
MEASUREMENT_COLORS = {
    "fixed_tagyfp_bgz_over_dapi_gate": "#c62828",
    "fixed_sox2_bgz_over_dapi_gate": "#b58900",
    "fixed_t_bgz_over_dapi_gate": "#2b6cb0",
}

CENTER_CANDIDATES_TSV = OUT_DIR / "array_center_candidates.tsv"
CENTER_SELECTED_TSV = OUT_DIR / "array_center_selected.tsv"
CENTER_METHOD_QC_PNG = ALTERNATE_FIGURE_DIR / "array_center_method_qc_examples.png"
CENTER_METHOD_LIVE_QC_PNG = ALTERNATE_FIGURE_DIR / "array_center_method_live_qc_examples.png"
CENTER_METHOD_AGREEMENT_PNG = ALTERNATE_FIGURE_DIR / "array_center_method_agreement.png"
CENTER_SELECTED_MONTAGE_PNG = ALTERNATE_FIGURE_DIR / "array_center_selected_montage.png"

CENTER_LIVE_PIXEL_BIN_STATS_TSV = OUT_DIR / "live_foxf1_center_distance_pixel_bin_stats.tsv"
CENTER_LIVE_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "live_foxf1_center_distance_trace_across_images.tsv"
CENTER_LIVE_TRACE_ALL_PIXELS_TSV = OUT_DIR / "live_foxf1_center_distance_trace_all_pixels_merged.tsv"
CENTER_LIVE_TRACE_EQUAL_SUPPORT_TSV = OUT_DIR / "live_foxf1_center_distance_trace_equal_support.tsv"
CENTER_LIVE_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV = OUT_DIR / "live_foxf1_center_distance_trace_equal_support_across_images.tsv"
CENTER_LIVE_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "live_foxf1_center_distance_plot_fixed_width.png"
CENTER_LIVE_TRACE_EQUAL_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "live_foxf1_center_distance_plot_equal_support.png"
CENTER_LIVE_SUPPORT_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "live_foxf1_center_support_fixed_width.png"
CENTER_LIVE_TRACE_EQUAL_SUPPORT_MERGED_PNG = ALTERNATE_FIGURE_DIR / "live_foxf1_center_all_pixels_merged_equal_support.png"
CENTER_LIVE_TRACE_FIXED_WIDTH_MERGED_PNG = ALTERNATE_FIGURE_DIR / "live_foxf1_center_all_pixels_merged_fixed_width.png"

CENTER_FIXED_PIXEL_BIN_STATS_TSV = OUT_DIR / "fixed_center_ratio_pixel_bin_stats.tsv"
CENTER_FIXED_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_center_ratio_distance_trace_across_images.tsv"
CENTER_FIXED_TRACE_ALL_PIXELS_TSV = OUT_DIR / "fixed_center_ratio_distance_trace_all_pixels_merged.tsv"
CENTER_FIXED_TRACE_EQUAL_SUPPORT_TSV = OUT_DIR / "fixed_center_ratio_distance_trace_equal_support.tsv"
CENTER_FIXED_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_center_ratio_distance_trace_equal_support_across_images.tsv"
CENTER_FIXED_TRACE_COMPARISON_PNG = ALTERNATE_FIGURE_DIR / "fixed_center_ratio_distance_trace_comparison.png"
CENTER_FIXED_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_center_ratio_distance_plot_fixed_width.png"
CENTER_FIXED_TRACE_EQUAL_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "fixed_center_ratio_distance_plot_equal_support.png"
CENTER_FIXED_SINGLE_CHANNEL_GRID_PNG = ALTERNATE_FIGURE_DIR / "fixed_center_single_channel_distance_plots.png"
CENTER_FIXED_SOX2_T_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_center_sox2_t_distance_plot_fixed_width.png"
CENTER_FIXED_SOX2_T_TRACE_EQUAL_SUPPORT_PNG = CANDIDATE_FIGURE_DIR / "fixed_center_sox2_t_distance_plot_equal_support.png"
SOX2_CENTER_DIP_REVIEW_PNG = ALTERNATE_FIGURE_DIR / "sox2_center_dip_annulus_review.png"
SOX2_CENTER_DIP_REVIEW_TSV = OUT_DIR / "sox2_center_dip_annulus_summary.tsv"


def _save_plot_all_formats(fig, png_path: Path, dpi: int = 180, bbox_inches: str = "tight") -> None:
    png_path = Path(png_path)
    fig.savefig(png_path, dpi=dpi, bbox_inches=bbox_inches)
    fig.savefig(png_path.with_suffix(".pdf"), bbox_inches=bbox_inches)
    fig.savefig(png_path.with_suffix(".svg"), bbox_inches=bbox_inches)


## Load Shared Intermediates

In [ ]:

required_outputs = [
    LIVE_FIXED_TSV,
    LIVE_MASK_SUMMARY_TSV,
    PER_IMAGE_BG_TSV,
    LIVE_FIELD_NPZ,
    LIVE_FIELD_SUMMARY_TSV,
    FIXED_GLOBAL_BG_TSV,
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError("Run 03, 05, and 07 first. Missing outputs:\n" + "\n".join(missing))

pos_df = common.load_position_manifest(POSITION_MANIFEST)

live_pos_df = common.filter_position_manifest(
    pos_df=pos_df,
    cohort_ids=[LIVE_COHORT_ID],
    conditions=["live"],
).sort_values(["canonical_position"]).reset_index(drop=True)
live_pos_df["canonical_position"] = live_pos_df["canonical_position"].astype(str)
live_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in live_pos_df.itertuples(index=False)}

fixed_pos_df = common.filter_position_manifest(
    pos_df=pos_df,
    cohort_ids=[FIXED_COHORT_ID],
    conditions=["fixed"],
).sort_values(["canonical_position"]).reset_index(drop=True)
fixed_pos_df["canonical_position"] = fixed_pos_df["canonical_position"].astype(str)
fixed_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in fixed_pos_df.itertuples(index=False)}

align_df = pd.read_csv(LIVE_FIXED_TSV, sep="	")
align_df["canonical_position"] = align_df["canonical_position"].astype(str)
align_df = align_df[align_df["status"].astype(str) == "ok"].copy()
align_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in align_df.itertuples(index=False)}

live_mask_summary_df = pd.read_csv(LIVE_MASK_SUMMARY_TSV, sep="	")
live_mask_summary_df["canonical_position"] = live_mask_summary_df["canonical_position"].astype(str)
per_image_bg_df = pd.read_csv(PER_IMAGE_BG_TSV, sep="	")
per_image_bg_df["canonical_position"] = per_image_bg_df["canonical_position"].astype(str)
live_field_summary_df = pd.read_csv(LIVE_FIELD_SUMMARY_TSV, sep="	")
fixed_global_bg_df = pd.read_csv(FIXED_GLOBAL_BG_TSV, sep="	")

with np.load(LIVE_FIELD_NPZ, allow_pickle=True) as payload:
    live_field = np.asarray(payload["field"], dtype=np.float32)

ok_live_positions = live_mask_summary_df.loc[live_mask_summary_df["status"] == "ok", "canonical_position"].astype(str).tolist()
live_masks_by_position = {}
for pos in ok_live_positions:
    path = LIVE_MASK_FINAL_DIR / f"{pos}_live_mask_from_fixed_final04.npz"
    if not path.exists():
        continue
    with np.load(path, allow_pickle=True) as payload:
        live_masks_by_position[str(pos)] = np.asarray(payload["mask"], dtype=bool)
ok_live_positions = [p for p in ok_live_positions if p in live_masks_by_position and p in align_by_pos]

final04 = lfq.build_final04_fixed_dapi_masks(
    position_manifest=POSITION_MANIFEST,
    cohort_id=FIXED_COHORT_ID,
    root=ROOT,
)
final04_summary_df = final04["summary_df"].copy()
final04_summary_df["canonical_position"] = final04_summary_df["canonical_position"].astype(str)
fixed_mask_payloads = final04["mask_payloads"]

print("Live positions with masks and alignment:", len(ok_live_positions))
print("Fixed positions with rebuilt final04 masks:", len(fixed_mask_payloads))
print("Live field shape:", tuple(live_field.shape))
display(final04_summary_df.head())
print()
display(fixed_global_bg_df.head())


## Helper Functions

In [ ]:

def load_live_bundle(canonical_position: str) -> dict:
    row = live_by_pos[str(canonical_position)]
    img = common.read_czi(ROOT / row["primary_analysis_file"])
    bf_idx = common.find_channel_index(img.channels, ["bright"])
    yfp_idx = common.find_channel_index(img.channels, LIVE_YFP_KEYWORDS)
    if bf_idx is None or yfp_idx is None:
        raise RuntimeError(f"Missing live BF or FOXF1-YFP reporter channel for {canonical_position}; channels={img.channels}")
    return {
        "row": row,
        "image": img,
        "bf": img.channel_images[int(bf_idx)].astype(np.float32),
        "yfp_raw": img.channel_images[int(yfp_idx)].astype(np.float32),
    }


def load_fixed_bundle(canonical_position: str) -> dict:
    row = fixed_by_pos[str(canonical_position)]
    img = lfq.read_czi_with_optional_fixed_small_plane_selection(
        ROOT / row["primary_analysis_file"],
        canonical_position=str(canonical_position),
    )
    bf_idx = common.find_channel_index(img.channels, ["bright"])
    dapi_idx = common.find_channel_index(img.channels, ["dapi"])
    yfp_idx = common.find_channel_index(img.channels, ["tagyfp", "foxf1", "yfp"])
    sox2_idx = common.find_channel_index(img.channels, ["568", "alexa fluor 568"])
    t_idx = common.find_channel_index(img.channels, ["647", "alexa fluor 647"])
    needed = [bf_idx, dapi_idx, yfp_idx, sox2_idx, t_idx]
    if any(v is None for v in needed):
        raise RuntimeError(f"Missing required fixed channels in {row['primary_analysis_file']}; channels={img.channels}")
    return {
        "row": row,
        "image": img,
        "bf": np.asarray(img.channel_images[int(bf_idx)], dtype=np.float32),
        "dapi_raw": np.asarray(img.channel_images[int(dapi_idx)], dtype=np.float32),
        "tagyfp_raw": np.asarray(img.channel_images[int(yfp_idx)], dtype=np.float32),
        "sox2_raw": np.asarray(img.channel_images[int(sox2_idx)], dtype=np.float32),
        "t_raw": np.asarray(img.channel_images[int(t_idx)], dtype=np.float32),
    }


def _largest_component(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask, dtype=bool)
    labels = measure.label(mask)
    if labels.max() <= 0:
        return mask
    props = measure.regionprops(labels)
    largest = max(props, key=lambda p: float(p.area)).label
    return np.asarray(labels == largest, dtype=bool)


def _mask_centroid_xy(mask: np.ndarray) -> tuple[float, float]:
    yy, xx = np.nonzero(np.asarray(mask, dtype=bool))
    if len(xx) == 0:
        return np.nan, np.nan
    return float(np.mean(xx)), float(np.mean(yy))


def _distance_peak_xy(mask: np.ndarray) -> tuple[float, float, float]:
    mask = np.asarray(mask, dtype=bool)
    if not np.any(mask):
        return np.nan, np.nan, np.nan
    dt = ndi.distance_transform_edt(mask)
    y, x = np.unravel_index(int(np.nanargmax(dt)), dt.shape)
    return float(x), float(y), float(dt[y, x])


def _longest_contour_xy(mask: np.ndarray) -> np.ndarray:
    contours = measure.find_contours(np.asarray(mask, dtype=np.float32), 0.5)
    if not contours:
        return np.zeros((0, 2), dtype=np.float64)
    cont = max(contours, key=lambda arr: int(arr.shape[0]))
    return np.c_[cont[:, 1], cont[:, 0]].astype(np.float64)


def _circle_fit_xy(boundary_xy: np.ndarray) -> tuple[float, float, float, float]:
    xy = np.asarray(boundary_xy, dtype=np.float64)
    if xy.ndim != 2 or xy.shape[0] < 8:
        return np.nan, np.nan, np.nan, np.nan
    x = xy[:, 0]
    y = xy[:, 1]
    A = np.c_[2.0 * x, 2.0 * y, np.ones_like(x)]
    b = x * x + y * y
    try:
        sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    except np.linalg.LinAlgError:
        return np.nan, np.nan, np.nan, np.nan
    cx, cy, c0 = sol
    radius = float(np.sqrt(max(c0 + cx * cx + cy * cy, 0.0)))
    resid = np.sqrt((x - cx) ** 2 + (y - cy) ** 2) - radius
    rmse = float(np.sqrt(np.mean(resid ** 2))) if resid.size else np.nan
    return float(cx), float(cy), radius, rmse


def _boundary_radial_cv(boundary_xy: np.ndarray, center_x: float, center_y: float) -> float:
    xy = np.asarray(boundary_xy, dtype=np.float64)
    if xy.ndim != 2 or xy.shape[0] < 8 or not np.isfinite(center_x) or not np.isfinite(center_y):
        return np.inf
    rr = np.sqrt((xy[:, 0] - float(center_x)) ** 2 + (xy[:, 1] - float(center_y)) ** 2)
    rr = rr[np.isfinite(rr)]
    if rr.size < 8:
        return np.inf
    mean_r = float(np.mean(rr))
    if mean_r <= float(lfq.EPS):
        return np.inf
    return float(np.std(rr) / mean_r)


def _snap_xy_to_mask(center_x: float, center_y: float, mask: np.ndarray) -> tuple[float, float]:
    mask = np.asarray(mask, dtype=bool)
    if not np.any(mask) or not np.isfinite(center_x) or not np.isfinite(center_y):
        return np.nan, np.nan
    ix = int(np.clip(np.round(center_x), 0, mask.shape[1] - 1))
    iy = int(np.clip(np.round(center_y), 0, mask.shape[0] - 1))
    if mask[iy, ix]:
        return float(center_x), float(center_y)
    yy, xx = np.nonzero(mask)
    if len(xx) == 0:
        return np.nan, np.nan
    d2 = (xx.astype(np.float64) - float(center_x)) ** 2 + (yy.astype(np.float64) - float(center_y)) ** 2
    idx = int(np.argmin(d2))
    return float(xx[idx]), float(yy[idx])


def _estimate_centers_from_round1_mask(round1_mask: np.ndarray) -> dict:
    center_mask = _largest_component(round1_mask)
    hull_mask = morphology.convex_hull_image(center_mask) if np.any(center_mask) else np.asarray(center_mask, dtype=bool)
    hull_boundary_xy = _longest_contour_xy(hull_mask)

    cx_cent, cy_cent = _mask_centroid_xy(center_mask)
    cx_dt, cy_dt, dt_radius = _distance_peak_xy(center_mask)
    cx_circ, cy_circ, circ_radius, circ_rmse = _circle_fit_xy(hull_boundary_xy)

    candidates = [
        {
            "method": "round1_centroid",
            "x_px": cx_cent,
            "y_px": cy_cent,
            "aux_value": np.nan,
            "radial_cv": _boundary_radial_cv(hull_boundary_xy, cx_cent, cy_cent),
        },
        {
            "method": "round1_dt_peak",
            "x_px": cx_dt,
            "y_px": cy_dt,
            "aux_value": dt_radius,
            "radial_cv": _boundary_radial_cv(hull_boundary_xy, cx_dt, cy_dt),
        },
        {
            "method": "round1_hull_circle_fit",
            "x_px": cx_circ,
            "y_px": cy_circ,
            "aux_value": circ_rmse,
            "radial_cv": _boundary_radial_cv(hull_boundary_xy, cx_circ, cy_circ),
        },
    ]
    valid = [c for c in candidates if np.isfinite(c["x_px"]) and np.isfinite(c["y_px"]) and np.isfinite(c["radial_cv"])]
    if valid:
        selected = min(valid, key=lambda c: float(c["radial_cv"]))
    else:
        selected = candidates[0]
    snap_x, snap_y = _snap_xy_to_mask(selected["x_px"], selected["y_px"], center_mask)
    return {
        "center_mask": np.asarray(center_mask, dtype=bool),
        "hull_mask": np.asarray(hull_mask, dtype=bool),
        "hull_boundary_xy": np.asarray(hull_boundary_xy, dtype=np.float64),
        "candidates": candidates,
        "selected_method": str(selected["method"]),
        "selected_x_px": float(snap_x),
        "selected_y_px": float(snap_y),
        "selected_unsnapped_x_px": float(selected["x_px"]),
        "selected_unsnapped_y_px": float(selected["y_px"]),
    }


def _center_distance_map_um(image_shape_yx: tuple[int, int], center_x_px: float, center_y_px: float, pixel_um_x: float, pixel_um_y: float) -> np.ndarray:
    h, w = image_shape_yx
    yy, xx = np.mgrid[0:h, 0:w]
    dist_um = np.sqrt(
        ((xx.astype(np.float32) - float(center_x_px)) * float(pixel_um_x)) ** 2 +
        ((yy.astype(np.float32) - float(center_y_px)) * float(pixel_um_y)) ** 2
    )
    return np.asarray(dist_um, dtype=np.float32)


def aggregate_trace_across_images(trace_df: pd.DataFrame, x_col: str = "bin_mid_um", y_col: str = "mean_value", unit_col: str = "canonical_position") -> pd.DataFrame:
    if len(trace_df) == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", x_col, "mean", "sd", "sem", "n_images"])
    df = trace_df[np.isfinite(trace_df[x_col]) & np.isfinite(trace_df[y_col])].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", x_col, "mean", "sd", "sem", "n_images"])
    rows = []
    group_cols = ["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", x_col]
    for keys, sub in df.groupby(group_cols, sort=True):
        vals = sub[y_col].to_numpy(dtype=float)
        n = int(sub[unit_col].astype(str).nunique())
        sd = float(np.nanstd(vals, ddof=1)) if n > 1 else 0.0
        sem = float(sd / np.sqrt(n)) if n > 1 else 0.0
        rows.append(
            {
                "measurement_name": str(keys[0]),
                "bin_idx": int(keys[1]),
                "bin_start_um": float(keys[2]),
                "bin_end_um": float(keys[3]),
                x_col: float(keys[4]),
                "mean": float(np.nanmean(vals)),
                "sd": sd,
                "sem": sem,
                "n_images": n,
            }
        )
    return pd.DataFrame(rows).sort_values(["measurement_name", x_col]).reset_index(drop=True)


def weighted_trace_pixels(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return pd.DataFrame(columns=[
            "measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um",
            "total_count_px", "total_weighted_sum", "weighted_mean_value", "pooled_sd_value", "pooled_sem_value"
        ])
    tmp = df.copy()
    counts = tmp["count_px"].astype(float)
    means = tmp["mean_value"].astype(float)
    stds = pd.to_numeric(tmp["std_value"], errors="coerce").astype(float)
    within_ss = np.where((counts > 1) & np.isfinite(stds), np.square(stds) * np.maximum(counts - 1.0, 0.0), 0.0)
    tmp["weighted_sum"] = means * counts
    tmp["sum_x2"] = within_ss + counts * np.square(means)
    agg = (
        tmp.groupby(["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um"], as_index=False)
        .agg(
            total_count_px=("count_px", "sum"),
            total_weighted_sum=("weighted_sum", "sum"),
            total_sum_x2=("sum_x2", "sum"),
        )
    )
    total_n = np.maximum(agg["total_count_px"].astype(float), 1.0)
    agg["weighted_mean_value"] = agg["total_weighted_sum"] / total_n
    numer = agg["total_sum_x2"] - np.square(agg["total_weighted_sum"]) / total_n
    denom = np.maximum(total_n - 1.0, 1.0)
    agg["pooled_sd_value"] = np.sqrt(np.maximum(numer / denom, 0.0))
    agg.loc[agg["total_count_px"].astype(float) <= 1.0, "pooled_sd_value"] = 0.0
    agg["pooled_sem_value"] = agg["pooled_sd_value"] / np.sqrt(total_n)
    return agg.sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)


def equal_support_trace(distance_um: np.ndarray, values: np.ndarray, pixels_per_bin: int, measurement_name: str) -> pd.DataFrame:
    dist = np.asarray(distance_um, dtype=np.float32)
    vals = np.asarray(values, dtype=np.float32)
    keep = np.isfinite(dist) & np.isfinite(vals)
    dist = dist[keep]
    vals = vals[keep]
    if dist.size == 0:
        return pd.DataFrame(columns=["measurement_name", "bin_idx", "bin_start_um", "bin_end_um", "bin_mid_um", "count_px", "mean_value", "median_value", "std_value", "sem_value"])
    order = np.argsort(dist, kind="mergesort")
    dist = dist[order]
    vals = vals[order]
    rows = []
    pixels_per_bin = max(1, int(pixels_per_bin))
    for start in range(0, len(dist), pixels_per_bin):
        stop = min(len(dist), start + pixels_per_bin)
        d = dist[start:stop]
        v = vals[start:stop]
        n = int(len(v))
        std = float(np.nanstd(v, ddof=1)) if n > 1 else 0.0
        sem = float(std / np.sqrt(n)) if n > 1 else 0.0
        rows.append(
            {
                "measurement_name": str(measurement_name),
                "bin_idx": int(len(rows)),
                "bin_start_um": float(np.nanmin(d)),
                "bin_end_um": float(np.nanmax(d)),
                "bin_mid_um": float(np.nanmean(d)),
                "count_px": n,
                "mean_value": float(np.nanmean(v)),
                "median_value": float(np.nanmedian(v)),
                "std_value": std,
                "sem_value": sem,
            }
        )
    return pd.DataFrame(rows)


def aggregate_equal_support_across_images(payloads: dict[str, dict[str, np.ndarray]], equal_support_df: pd.DataFrame, measurement_name: str) -> pd.DataFrame:
    rows = []
    for _, rr in equal_support_df.sort_values("bin_idx").iterrows():
        lo = float(rr["bin_start_um"])
        hi = float(rr["bin_end_um"])
        image_means = []
        image_counts = []
        for pos, payload in payloads.items():
            dist = np.asarray(payload["distance_um"], dtype=np.float32)
            vals = np.asarray(payload["value"], dtype=np.float32)
            keep = np.isfinite(dist) & np.isfinite(vals) & (dist >= lo - 1e-9) & (dist <= hi + 1e-9)
            if not np.any(keep):
                continue
            vv = vals[keep]
            image_means.append(float(np.nanmean(vv)))
            image_counts.append(int(vv.size))
        arr = np.asarray(image_means, dtype=float)
        n = int(np.sum(np.isfinite(arr)))
        mean = float(np.nanmean(arr)) if n else np.nan
        sd = float(np.nanstd(arr, ddof=1)) if n > 1 else (0.0 if n == 1 else np.nan)
        sem = float(sd / np.sqrt(n)) if n > 1 else (0.0 if n == 1 else np.nan)
        rows.append(
            {
                "measurement_name": str(measurement_name),
                "bin_idx": int(rr["bin_idx"]),
                "bin_start_um": lo,
                "bin_end_um": hi,
                "bin_mid_um": float(rr["bin_mid_um"]),
                "bin_width_um": float(hi - lo),
                "total_count_px": int(np.sum(image_counts)) if image_counts else 0,
                "n_images": n,
                "mean": mean,
                "sd": sd,
                "sem": sem,
            }
        )
    return pd.DataFrame(rows).sort_values("bin_idx").reset_index(drop=True)


def _trace_shared_ylim(main_agg: pd.DataFrame, alt_agg: pd.DataFrame):
    ymins = []
    ymaxs = []
    for agg_df, mean_col, sd_col in [(main_agg, "mean", "sd"), (alt_agg, "mean", "sd")]:
        if len(agg_df) > 0:
            ymins.append(float(np.nanmin(agg_df[mean_col] - agg_df[sd_col])))
            ymaxs.append(float(np.nanmax(agg_df[mean_col] + agg_df[sd_col])))
    if ymins and ymaxs:
        y_lo = float(np.nanmin(ymins))
        y_hi = float(np.nanmax(ymaxs))
        y_span = max(1e-6, y_hi - y_lo)
        return (y_lo - 0.06 * y_span, y_hi + 0.02 * y_span)
    return None


def _normalized_trace_inputs(per_image_df: pd.DataFrame, agg_df: pd.DataFrame, mean_col: str, sd_col: str):
    per_norm = per_image_df.copy()
    agg_norm = agg_df.copy()
    norm_rows = []
    for meas_name in MEASUREMENT_ORDER:
        sub = agg_norm[agg_norm["measurement_name"] == meas_name].copy()
        vals = sub[mean_col].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        peak = float(np.nanmax(vals)) if vals.size else 1.0
        if not np.isfinite(peak) or peak <= float(lfq.EPS):
            peak = 1.0
        norm_rows.append({"measurement_name": meas_name, "plot_peak_scale": peak})
        per_mask = per_norm["measurement_name"] == meas_name
        agg_mask = agg_norm["measurement_name"] == meas_name
        for col in ["mean_value", "median_value", "std_value", "sem_value"]:
            if col in per_norm.columns:
                per_norm.loc[per_mask, col] = per_norm.loc[per_mask, col].astype(float) / peak
        for col in [mean_col, sd_col, "sem"]:
            if col in agg_norm.columns:
                agg_norm.loc[agg_mask, col] = agg_norm.loc[agg_mask, col].astype(float) / peak
    return per_norm, agg_norm, pd.DataFrame(norm_rows)


def _interval_band_arrays(df: pd.DataFrame, mean_col: str, band_col: str | None = None):
    sub = df.sort_values("bin_mid_um").copy()
    xs: list[float] = []
    means: list[float] = []
    lows: list[float] = []
    highs: list[float] = []
    use_band = band_col is not None and band_col in sub.columns
    for _, rr in sub.iterrows():
        x0 = float(rr["bin_start_um"])
        x1 = float(rr["bin_end_um"])
        yy = float(rr[mean_col])
        if not (np.isfinite(x0) and np.isfinite(x1) and np.isfinite(yy)):
            continue
        xs.extend([x0, x1])
        means.extend([yy, yy])
        if use_band:
            bb = float(rr[band_col])
            if not np.isfinite(bb):
                bb = 0.0
            lows.extend([yy - bb, yy - bb])
            highs.extend([yy + bb, yy + bb])
    return np.asarray(xs, dtype=float), np.asarray(means, dtype=float), np.asarray(lows, dtype=float), np.asarray(highs, dtype=float)


def _plot_live_distance_trace(ax, per_image_df: pd.DataFrame, agg_df: pd.DataFrame, x_col: str, y_col: str, sd_col: str, title: str, color: str = "#c62828", span_bins: bool = False):
    for _, sub in per_image_df.groupby("canonical_position", sort=True):
        sub = sub.sort_values("bin_mid_um")
        ax.plot(sub["bin_mid_um"], sub["mean_value"], color="0.65", alpha=0.14, linewidth=0.8)
        ax.scatter(sub["bin_mid_um"], sub["mean_value"], color="0.65", alpha=0.09, s=6)
    if len(agg_df) > 0:
        agg_df = agg_df.sort_values(x_col)
        mean_vals = agg_df[y_col].to_numpy(dtype=float)
        sd_vals = agg_df[sd_col].to_numpy(dtype=float)
        if span_bins and {"bin_start_um", "bin_end_um"}.issubset(agg_df.columns):
            xs, yy, lo, hi = _interval_band_arrays(agg_df, mean_col=y_col, band_col=sd_col)
            ax.plot(xs, yy, color=color, linewidth=2.6, label="Mean trace")
            ax.fill_between(xs, lo, hi, color=color, alpha=0.22, label="±SD")
            ax.scatter(agg_df[x_col], mean_vals, color=color, s=12, alpha=0.85, zorder=4)
        else:
            ax.plot(agg_df[x_col], mean_vals, color=color, linewidth=2.6, label="Mean trace")
            ax.fill_between(agg_df[x_col], mean_vals - sd_vals, mean_vals + sd_vals, color=color, alpha=0.22, label="±SD")
    ax.set_xlabel("Distance from selected array center (um)")
    ax.set_ylabel("Mean FOXF1-YFP reporter activity (live; flat-field corrected, per-image bg-subtracted)")
    ax.set_title(title)
    ax.legend(loc="best", frameon=False)


def _plot_multi_channel_trace(ax, per_image_df: pd.DataFrame, agg_df: pd.DataFrame, measurement_order: list[str], title: str, ylabel: str, span_bins: bool = False) -> None:
    for meas_name in measurement_order:
        color = MEASUREMENT_COLORS[meas_name]
        label = MEASUREMENT_LABELS[meas_name]
        per_sub = per_image_df[per_image_df["measurement_name"] == meas_name].copy()
        agg_sub = agg_df[agg_df["measurement_name"] == meas_name].copy()
        for _, sub in per_sub.groupby("canonical_position", sort=True):
            sub = sub.sort_values("bin_mid_um")
            ax.plot(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.10, linewidth=0.8)
            ax.scatter(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.06, s=6)
        if len(agg_sub) > 0:
            agg_sub = agg_sub.sort_values("bin_mid_um")
            mean_vals = agg_sub["mean"].to_numpy(dtype=float)
            sd_vals = agg_sub["sd"].to_numpy(dtype=float)
            if span_bins and {"bin_start_um", "bin_end_um"}.issubset(agg_sub.columns):
                xs, yy, lo, hi = _interval_band_arrays(agg_sub, mean_col="mean", band_col="sd")
                ax.plot(xs, yy, color=color, linewidth=2.4, label=label)
                ax.fill_between(xs, lo, hi, color=color, alpha=0.18)
                ax.scatter(agg_sub["bin_mid_um"], mean_vals, color=color, alpha=0.85, s=10, zorder=4)
            else:
                ax.plot(agg_sub["bin_mid_um"], mean_vals, color=color, linewidth=2.4, label=label)
                ax.fill_between(agg_sub["bin_mid_um"], mean_vals - sd_vals, mean_vals + sd_vals, color=color, alpha=0.18)
    ax.set_xlabel("Distance from selected array center (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best", frameon=False)


def _plot_multi_channel_merged(ax, merged_df: pd.DataFrame, measurement_order: list[str], title: str, ylabel: str, mean_col: str, sd_col: str, count_col: str | None = None, add_ci95: bool = True) -> None:
    for meas_name in measurement_order:
        color = MEASUREMENT_COLORS[meas_name]
        label = MEASUREMENT_LABELS[meas_name]
        sub = merged_df[merged_df["measurement_name"] == meas_name].copy().sort_values("bin_mid_um")
        if len(sub) == 0:
            continue
        mean_vals = sub[mean_col].to_numpy(dtype=float)
        sd_vals = sub[sd_col].to_numpy(dtype=float)
        ax.plot(sub["bin_mid_um"], mean_vals, color=color, linewidth=2.3, label=label)
        ax.fill_between(sub["bin_mid_um"], mean_vals - sd_vals, mean_vals + sd_vals, color=color, alpha=0.16)
        if add_ci95 and count_col is not None and count_col in sub.columns:
            n = np.maximum(sub[count_col].to_numpy(dtype=float), 1.0)
            ci95 = 1.96 * sd_vals / np.sqrt(n)
            ax.plot(sub["bin_mid_um"], mean_vals - ci95, color=color, lw=1.2, ls="--", alpha=0.9)
            ax.plot(sub["bin_mid_um"], mean_vals + ci95, color=color, lw=1.2, ls="--", alpha=0.9)
    ax.set_xlabel("Distance from selected array center (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="upper left", frameon=False)


def _plot_single_fixed_channel_trace(ax, per_image_df: pd.DataFrame, agg_df: pd.DataFrame, meas_name: str, title: str, ylabel: str, color: str):
    per_sub = per_image_df[per_image_df["measurement_name"] == meas_name].copy().sort_values(["canonical_position", "bin_mid_um"])
    agg_sub = agg_df[agg_df["measurement_name"] == meas_name].copy().sort_values("bin_mid_um")
    first_gray = True
    for _, sub in per_sub.groupby("canonical_position", sort=True):
        sub = sub.sort_values("bin_mid_um")
        ax.plot(sub["bin_mid_um"], sub["mean_value"], color="0.75", lw=0.8, alpha=0.45, zorder=1, label="Per-image traces" if first_gray else None)
        ax.scatter(sub["bin_mid_um"], sub["mean_value"], color="0.75", s=6, alpha=0.09, zorder=1)
        first_gray = False
    if len(agg_sub) > 0:
        mean_vals = agg_sub["mean"].to_numpy(dtype=float)
        sd_vals = agg_sub["sd"].to_numpy(dtype=float)
        ax.plot(agg_sub["bin_mid_um"], mean_vals, color=color, lw=2.4, zorder=3, label=MEASUREMENT_LABELS[meas_name])
        ax.fill_between(agg_sub["bin_mid_um"], mean_vals - sd_vals, mean_vals + sd_vals, color=color, alpha=0.18, zorder=2, label="Mean ± 95% CI across images")
    ax.set_xlabel("Distance from selected array center (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc="best")


def _peak_within_window(df: pd.DataFrame, mean_col: str, xmin: float = 0.0, xmax: float = np.inf, measurement_name: str | None = None, x_col: str = "bin_mid_um") -> float:
    sub = df.copy()
    if measurement_name is not None and "measurement_name" in sub.columns:
        sub = sub[sub["measurement_name"].astype(str) == str(measurement_name)].copy()
    if x_col in sub.columns:
        xx = pd.to_numeric(sub[x_col], errors="coerce").astype(float)
        sub = sub[np.isfinite(xx) & xx.between(float(xmin), float(xmax), inclusive="both")].copy()
    vals = pd.to_numeric(sub[mean_col], errors="coerce").to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    peak = float(np.nanmax(vals)) if vals.size else 1.0
    if not np.isfinite(peak) or peak <= float(lfq.EPS):
        peak = 1.0
    return peak


def _windowed_ylim(df: pd.DataFrame, mean_col: str, band_col: str, xmin: float = 0.0, xmax: float = np.inf, measurement_name: str | None = None, x_col: str = "bin_mid_um", floor: float | None = None, ceil_min: float | None = None) -> tuple[float, float]:
    sub = df.copy()
    if measurement_name is not None and "measurement_name" in sub.columns:
        sub = sub[sub["measurement_name"].astype(str) == str(measurement_name)].copy()
    xx = pd.to_numeric(sub[x_col], errors="coerce").astype(float)
    sub = sub[np.isfinite(xx) & xx.between(float(xmin), float(xmax), inclusive="both")].copy()
    if len(sub) == 0:
        return (floor if floor is not None else -0.2, ceil_min if ceil_min is not None else 1.2)
    mean_vals = pd.to_numeric(sub[mean_col], errors="coerce").to_numpy(dtype=float)
    band_vals = pd.to_numeric(sub[band_col], errors="coerce").to_numpy(dtype=float)
    finite = np.isfinite(mean_vals) & np.isfinite(band_vals)
    mean_vals = mean_vals[finite]
    band_vals = band_vals[finite]
    if mean_vals.size == 0:
        return (floor if floor is not None else -0.2, ceil_min if ceil_min is not None else 1.2)
    lo = float(np.nanmin(mean_vals - band_vals))
    hi = float(np.nanmax(mean_vals + band_vals))
    lo = float(np.floor((lo - 0.05) / 0.05) * 0.05)
    hi = float(np.ceil((hi + 0.05) / 0.05) * 0.05)
    if floor is not None:
        lo = max(float(floor), lo)
    if ceil_min is not None:
        hi = max(float(ceil_min), hi)
    if hi <= lo:
        hi = lo + 0.2
    return (lo, hi)


def _apply_anterior_axis(ax, xmin: float, xmax: float, x_label: str = CENTER_TRACE_XLABEL) -> None:
    ax.set_xlim(float(xmax), float(xmin))
    ax.set_xlabel(x_label)


def _normalized_trace_inputs(per_image_df: pd.DataFrame, agg_df: pd.DataFrame, mean_col: str, sd_col: str, measurement_order: list[str] | None = None, peak_xmin: float = 0.0, peak_xmax: float = np.inf):
    per_norm = per_image_df.copy()
    agg_norm = agg_df.copy()
    if measurement_order is None:
        measurement_order = agg_norm["measurement_name"].astype(str).drop_duplicates().tolist() if "measurement_name" in agg_norm.columns else []
    norm_rows = []
    for meas_name in measurement_order:
        sub = agg_norm[agg_norm["measurement_name"] == meas_name].copy() if "measurement_name" in agg_norm.columns else agg_norm.copy()
        peak = _peak_within_window(sub, mean_col=mean_col, xmin=peak_xmin, xmax=peak_xmax)
        norm_rows.append({"measurement_name": meas_name, "plot_peak_scale": peak})
        if "measurement_name" in per_norm.columns:
            per_mask = per_norm["measurement_name"] == meas_name
            for col in ["mean_value", "median_value", "std_value", "sem_value"]:
                if col in per_norm.columns:
                    per_norm.loc[per_mask, col] = pd.to_numeric(per_norm.loc[per_mask, col], errors="coerce").astype(float) / peak
        if "measurement_name" in agg_norm.columns:
            agg_mask = agg_norm["measurement_name"] == meas_name
            for col in [mean_col, sd_col, "sem"]:
                if col in agg_norm.columns:
                    agg_norm.loc[agg_mask, col] = pd.to_numeric(agg_norm.loc[agg_mask, col], errors="coerce").astype(float) / peak
    return per_norm, agg_norm, pd.DataFrame(norm_rows)


## Estimate Array Centers

In [ ]:

center_rows = []
center_payloads = {}
for pos in fixed_pos_df["canonical_position"].astype(str).tolist():
    if str(pos) not in fixed_mask_payloads or str(pos) not in align_by_pos:
        continue
    payload = fixed_mask_payloads[str(pos)]
    round1_mask = np.asarray(payload["round1_core_mask"], dtype=bool)
    est = _estimate_centers_from_round1_mask(round1_mask)

    align_row = align_by_pos[str(pos)]
    fixed_to_live = lfa.affine_matrix_from_row_prefix(align_row, "fixed_small_to_live_small")
    live_x, live_y = lfa.apply_affine_to_xy(est["selected_x_px"], est["selected_y_px"], fixed_to_live)

    row = {
        "canonical_position": str(pos),
        "selected_method": str(est["selected_method"]),
        "selected_fixed_x_px": float(est["selected_x_px"]),
        "selected_fixed_y_px": float(est["selected_y_px"]),
        "selected_live_x_px": float(live_x),
        "selected_live_y_px": float(live_y),
        "round1_area_px": int(np.sum(est["center_mask"])),
        "hull_area_px": int(np.sum(est["hull_mask"])),
    }
    for cand in est["candidates"]:
        method = str(cand["method"])
        row[f"{method}_x_px"] = float(cand["x_px"])
        row[f"{method}_y_px"] = float(cand["y_px"])
        row[f"{method}_aux"] = float(cand["aux_value"]) if np.isfinite(cand["aux_value"]) else np.nan
        row[f"{method}_radial_cv"] = float(cand["radial_cv"]) if np.isfinite(cand["radial_cv"]) else np.nan
    center_rows.append(row)
    center_payloads[str(pos)] = {
        **est,
        "selected_live_x_px": float(live_x),
        "selected_live_y_px": float(live_y),
        "dapi_max": np.asarray(payload["dapi_max"], dtype=np.float32),
        "selected_plane": np.asarray(payload["selected_plane"], dtype=np.float32),
        "round1_mask": np.asarray(round1_mask, dtype=bool),
    }

center_df = pd.DataFrame(center_rows).sort_values("canonical_position").reset_index(drop=True)
center_df.to_csv(CENTER_SELECTED_TSV, sep="	", index=False)

cand_long_rows = []
for _, rr in center_df.iterrows():
    for method in CENTER_METHOD_ORDER:
        cand_long_rows.append(
            {
                "canonical_position": str(rr["canonical_position"]),
                "method": method,
                "x_px": float(rr.get(f"{method}_x_px", np.nan)),
                "y_px": float(rr.get(f"{method}_y_px", np.nan)),
                "aux_value": float(rr.get(f"{method}_aux", np.nan)),
                "radial_cv": float(rr.get(f"{method}_radial_cv", np.nan)),
                "selected_method": str(rr["selected_method"]),
            }
        )
center_candidates_df = pd.DataFrame(cand_long_rows)
center_candidates_df.to_csv(CENTER_CANDIDATES_TSV, sep="	", index=False)

print("Estimated array centers:", len(center_df))
print("Selected methods:")
print(center_df["selected_method"].value_counts(dropna=False).to_string())
display(center_df.head())


## Center QC

In [ ]:

rep_positions = [p for p in CENTER_REPRESENTATIVE_POSITIONS if p in center_payloads and p in live_masks_by_position]
fig, axes = plt.subplots(len(rep_positions), 2, figsize=(12.5, 4.6 * len(rep_positions)), constrained_layout=True)
if len(rep_positions) == 1:
    axes = np.asarray([axes])
for row_idx, pos in enumerate(rep_positions):
    cp = center_payloads[str(pos)]
    fixed_ax = axes[row_idx, 0]
    fixed_ax.imshow(common._robust_rescale(cp["dapi_max"]), cmap="magma")
    fixed_ax.contour(cp["round1_mask"].astype(np.float32), levels=[0.5], colors=["gold"], linewidths=0.9)
    fixed_ax.contour(cp["hull_mask"].astype(np.float32), levels=[0.5], colors=["cyan"], linewidths=0.8)
    for cand in cp["candidates"]:
        fixed_ax.scatter(cand["x_px"], cand["y_px"], s=44, color=CENTER_METHOD_COLORS[cand["method"]], edgecolors="black", linewidths=0.4)
    fixed_ax.scatter(cp["selected_x_px"], cp["selected_y_px"], s=88, color=CENTER_METHOD_COLORS["selected"], edgecolors="white", linewidths=1.0)
    fixed_ax.set_title(f"{pos} | Fixed DAPI + round1 mask + center candidates")
    fixed_ax.axis("off")

    live_ctx = load_live_bundle(pos)
    live_mask = np.asarray(live_masks_by_position[str(pos)], dtype=bool)
    live_ax = axes[row_idx, 1]
    live_ax.imshow(common._robust_rescale(live_ctx["bf"]), cmap="gray")
    live_ax.contour(live_mask.astype(np.float32), levels=[0.5], colors=["lime"], linewidths=0.9)
    live_ax.scatter(cp["selected_live_x_px"], cp["selected_live_y_px"], s=88, color=CENTER_METHOD_COLORS["selected"], edgecolors="white", linewidths=1.0)
    live_ax.set_title(f"{pos} | Live BF + transferred mask + mapped selected center")
    live_ax.axis("off")
_save_plot_all_formats(fig, CENTER_METHOD_QC_PNG, dpi=180, bbox_inches="tight")
plt.show()

pair_rows = []
for _, rr in center_df.iterrows():
    for a in CENTER_METHOD_ORDER:
        for b in CENTER_METHOD_ORDER:
            if a >= b:
                continue
            ax = float(rr.get(f"{a}_x_px", np.nan)); ay = float(rr.get(f"{a}_y_px", np.nan))
            bx = float(rr.get(f"{b}_x_px", np.nan)); by = float(rr.get(f"{b}_y_px", np.nan))
            if np.isfinite(ax) and np.isfinite(ay) and np.isfinite(bx) and np.isfinite(by):
                pair_rows.append({
                    "canonical_position": str(rr["canonical_position"]),
                    "pair": f"{a} vs {b}",
                    "distance_px": float(np.sqrt((ax - bx) ** 2 + (ay - by) ** 2)),
                })
pair_df = pd.DataFrame(pair_rows)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), constrained_layout=True)
if len(pair_df) > 0:
    for pair, sub in pair_df.groupby("pair", sort=True):
        axes[0].hist(sub["distance_px"].astype(float), bins=18, alpha=0.55, label=pair)
    axes[0].set_xlabel("Pairwise center disagreement (px)")
    axes[0].set_ylabel("Position count")
    axes[0].set_title("Agreement between center methods")
    axes[0].legend(frameon=False, fontsize=8)
method_counts = center_df["selected_method"].value_counts(dropna=False)
axes[1].bar(method_counts.index.astype(str), method_counts.values.astype(float), color=[CENTER_METHOD_COLORS.get(k, "0.6") for k in method_counts.index.astype(str)])
axes[1].set_ylabel("Selected positions")
axes[1].set_title("Selected center method across positions")
axes[1].tick_params(axis="x", rotation=25)
_save_plot_all_formats(fig, CENTER_METHOD_AGREEMENT_PNG, dpi=180, bbox_inches="tight")
plt.show()


### Selected Centers For All Positions


In [ ]:
all_positions = center_df["canonical_position"].astype(str).tolist()
n_cols = 6
n_rows = int(np.ceil(len(all_positions) / n_cols)) if len(all_positions) else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows), constrained_layout=True)
axes = np.asarray(axes).reshape(n_rows, n_cols)
for ax in axes.ravel():
    ax.axis("off")
for idx, pos in enumerate(all_positions):
    ax = axes.ravel()[idx]
    cp = center_payloads[str(pos)]
    ax.imshow(common._robust_rescale(cp["dapi_max"]), cmap="magma")
    ax.contour(cp["round1_mask"].astype(np.float32), levels=[0.5], colors=["gold"], linewidths=0.6)
    ax.scatter(cp["selected_x_px"], cp["selected_y_px"], s=78, facecolors="white", edgecolors="black", linewidths=1.1)
    ax.text(0.03, 0.97, str(pos), transform=ax.transAxes, ha="left", va="top", fontsize=8.5, color="white", bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.55, "boxstyle": "round,pad=0.18"})
    ax.axis("off")
_save_plot_all_formats(fig, CENTER_SELECTED_MONTAGE_PNG, dpi=180, bbox_inches="tight")
plt.show()


## Live FOXF1 Vs Center Distance

In [ ]:

per_image_mu_map = per_image_bg_df.set_index("canonical_position")["mu_bg_raw"].astype(float).to_dict()

live_rows = []
live_pixel_payloads = {}
for pos in ok_live_positions:
    if str(pos) not in center_df.set_index("canonical_position").index:
        continue
    live_ctx = load_live_bundle(pos)
    live_mask = np.asarray(live_masks_by_position[str(pos)], dtype=bool)
    corrected = lfq.apply_illumination_field(live_ctx["yfp_raw"], live_field)
    mu_i = float(per_image_mu_map[str(pos)])
    bgsub = common.subtract_uniform_background(
        image=corrected,
        mu_bg_raw=mu_i,
        clip_below_zero=True,
    )
    rr = center_df.set_index("canonical_position").loc[str(pos)]
    dist_um = _center_distance_map_um(
        image_shape_yx=bgsub.shape,
        center_x_px=float(rr["selected_live_x_px"]),
        center_y_px=float(rr["selected_live_y_px"]),
        pixel_um_x=float(live_ctx["image"].pixel_um_x),
        pixel_um_y=float(live_ctx["image"].pixel_um_y),
    )
    stats = lfq.masked_distance_bin_stats(
        value_img=bgsub,
        distance_um_map=dist_um,
        mask=live_mask,
        bin_um=float(CENTER_BIN_UM),
    )
    for _, tr in stats.iterrows():
        live_rows.append(
            {
                "canonical_position": str(pos),
                "image_id": str(live_ctx["row"].get("image_id", f"{LIVE_COHORT_ID}_{pos}")),
                "cohort_id": LIVE_COHORT_ID,
                "measurement_name": "live_tagyfp_flatfield_per_image_bgsub_center",
                "bin_idx": int(tr["bin_idx"]),
                "bin_start_um": float(tr["bin_start_um"]),
                "bin_end_um": float(tr["bin_end_um"]),
                "bin_mid_um": float(tr["bin_mid_um"]),
                "count_px": int(tr["count_px"]),
                "mean_value": float(tr["mean_value"]),
                "median_value": float(tr["median_value"]),
                "std_value": float(tr["std_value"]) if pd.notna(tr["std_value"]) else np.nan,
                "sem_value": float(tr["sem_value"]) if pd.notna(tr["sem_value"]) else np.nan,
            }
        )
    keep = live_mask & np.isfinite(bgsub) & np.isfinite(dist_um)
    live_pixel_payloads[str(pos)] = {
        "distance_um": np.asarray(dist_um[keep], dtype=np.float32),
        "value": np.asarray(bgsub[keep], dtype=np.float32),
    }

live_center_stats_df = pd.DataFrame(live_rows).sort_values(["canonical_position", "bin_idx"]).reset_index(drop=True)
live_center_stats_df.to_csv(CENTER_LIVE_PIXEL_BIN_STATS_TSV, sep="	", index=False)

live_center_trace_df = aggregate_trace_across_images(live_center_stats_df)
live_center_trace_df.to_csv(CENTER_LIVE_TRACE_ACROSS_IMAGES_TSV, sep="	", index=False)

live_center_merged_df = weighted_trace_pixels(live_center_stats_df)
live_center_merged_df.to_csv(CENTER_LIVE_TRACE_ALL_PIXELS_TSV, sep="	", index=False)

live_dist_all = np.concatenate([v["distance_um"] for v in live_pixel_payloads.values() if len(v["distance_um"])]) if live_pixel_payloads else np.array([], dtype=np.float32)
live_val_all = np.concatenate([v["value"] for v in live_pixel_payloads.values() if len(v["value"])]) if live_pixel_payloads else np.array([], dtype=np.float32)
live_equal_support_target_px = int(np.nanmedian(live_center_merged_df["total_count_px"].astype(float))) if len(live_center_merged_df) else 1
live_center_equal_support_df = equal_support_trace(
    live_dist_all,
    live_val_all,
    pixels_per_bin=live_equal_support_target_px,
    measurement_name="live_tagyfp_flatfield_per_image_bgsub_center_equal_support",
)
live_center_equal_support_df.to_csv(CENTER_LIVE_TRACE_EQUAL_SUPPORT_TSV, sep="	", index=False)

live_center_equal_support_across_images_df = aggregate_equal_support_across_images(
    live_pixel_payloads,
    live_center_equal_support_df,
    measurement_name="live_tagyfp_flatfield_per_image_bgsub_center_equal_support_across_images",
)
live_center_equal_support_across_images_df.to_csv(CENTER_LIVE_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV, sep="	", index=False)

main_agg_raw = live_center_trace_df.sort_values("bin_mid_um").copy()
alt_agg_raw = live_center_equal_support_across_images_df.sort_values("bin_mid_um").copy()

live_fixedwidth_scale = _peak_within_window(main_agg_raw, mean_col="mean", xmin=CENTER_TRACE_FIXEDWIDTH_XMIN, xmax=CENTER_TRACE_XMAX)
live_equal_support_scale = _peak_within_window(alt_agg_raw, mean_col="mean", xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN, xmax=CENTER_TRACE_XMAX)

def _scale_live_per_image(df: pd.DataFrame, scale: float) -> pd.DataFrame:
    out = df.copy()
    for col in ["mean_value", "median_value", "std_value", "sem_value"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype(float) / float(scale)
    return out

def _scale_live_agg(df: pd.DataFrame, scale: float) -> pd.DataFrame:
    out = df.copy()
    for col in ["mean", "sd", "sem"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype(float) / float(scale)
    return out

live_plot_fixedwidth_per_df = _scale_live_per_image(live_center_stats_df, live_fixedwidth_scale)
live_plot_equal_support_per_df = _scale_live_per_image(live_center_stats_df, live_equal_support_scale)
live_plot_main_agg = _scale_live_agg(main_agg_raw, live_fixedwidth_scale)
live_plot_alt_agg = _scale_live_agg(alt_agg_raw, live_equal_support_scale)

live_plot_fixedwidth_merged = live_center_merged_df.copy()
fw_peak = _peak_within_window(live_center_merged_df.rename(columns={"weighted_mean_value": "mean_tmp"}), mean_col="mean_tmp", xmin=CENTER_TRACE_FIXEDWIDTH_XMIN, xmax=CENTER_TRACE_XMAX)
for col in ["weighted_mean_value", "pooled_sd_value", "pooled_sem_value"]:
    if col in live_plot_fixedwidth_merged.columns:
        live_plot_fixedwidth_merged[col] = pd.to_numeric(live_plot_fixedwidth_merged[col], errors="coerce").astype(float) / fw_peak

live_plot_equal_support_merged = live_center_equal_support_df.copy()
eq_peak = _peak_within_window(live_center_equal_support_df, mean_col="mean_value", xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN, xmax=CENTER_TRACE_XMAX)
for col in ["mean_value", "median_value", "std_value", "sem_value"]:
    if col in live_plot_equal_support_merged.columns:
        live_plot_equal_support_merged[col] = pd.to_numeric(live_plot_equal_support_merged[col], errors="coerce").astype(float) / eq_peak

live_fixedwidth_ylim = _windowed_ylim(live_plot_main_agg, mean_col="mean", band_col="sd", xmin=CENTER_TRACE_FIXEDWIDTH_XMIN, xmax=CENTER_TRACE_XMAX, floor=0.0, ceil_min=1.0)
live_equal_support_ylim = _windowed_ylim(live_plot_alt_agg, mean_col="mean", band_col="sd", xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN, xmax=CENTER_TRACE_XMAX, floor=0.0, ceil_min=1.0)
live_fixedwidth_merged_ylim = _windowed_ylim(live_plot_fixedwidth_merged.rename(columns={"weighted_mean_value": "mean_tmp", "pooled_sd_value": "sd_tmp"}), mean_col="mean_tmp", band_col="sd_tmp", xmin=CENTER_TRACE_FIXEDWIDTH_XMIN, xmax=CENTER_TRACE_XMAX, floor=0.0, ceil_min=1.0)
live_equal_support_merged_ylim = _windowed_ylim(live_plot_equal_support_merged, mean_col="mean_value", band_col="std_value", xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN, xmax=CENTER_TRACE_XMAX, floor=0.0, ceil_min=1.0)

support_sub = live_center_merged_df[live_center_merged_df["bin_mid_um"].astype(float) <= CENTER_TRACE_XMAX].copy()
support_ymax = float(np.ceil(np.nanmax(support_sub["total_count_px"].astype(float)) * 1.05 / 1000.0) * 1000.0) if len(support_sub) else 1.0
support_ymax = max(support_ymax, 1.0)

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_live_distance_trace(
    ax=ax,
    per_image_df=live_plot_equal_support_per_df,
    agg_df=live_plot_alt_agg,
    x_col="bin_mid_um",
    y_col="mean",
    sd_col="sd",
    title="Live FOXF1-YFP reporter activity by distance from anterior | equal-support bins (peak-normalized)",
    color=MEASUREMENT_COLORS["fixed_tagyfp_bgz_over_dapi_gate"],
)
_apply_anterior_axis(ax, CENTER_TRACE_EQUAL_SUPPORT_XMIN, CENTER_TRACE_XMAX)
ax.set_ylim(*live_equal_support_ylim)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_LIVE_TRACE_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_live_distance_trace(
    ax=ax,
    per_image_df=live_plot_fixedwidth_per_df,
    agg_df=live_plot_main_agg,
    x_col="bin_mid_um",
    y_col="mean",
    sd_col="sd",
    title="Live FOXF1-YFP reporter activity by distance from anterior | fixed-width bins (peak-normalized)",
    color=MEASUREMENT_COLORS["fixed_tagyfp_bgz_over_dapi_gate"],
)
_apply_anterior_axis(ax, CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)
ax.set_ylim(*live_fixedwidth_ylim)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_LIVE_TRACE_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 4.6))
bar_widths = (support_sub["bin_end_um"] - support_sub["bin_start_um"]).astype(float).to_numpy() if len(support_sub) else np.array([])
ax.bar(support_sub["bin_mid_um"], support_sub["total_count_px"], width=bar_widths, color="0.7", edgecolor="0.45", linewidth=0.5, align="center")
ax.set_title("Transferred-mask support | total retained pixels pooled across images, fixed-width anterior-distance bins")
_apply_anterior_axis(ax, CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)
ax.set_ylabel("Total masked pixels in bin")
ax.set_ylim(0.0, support_ymax)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_LIVE_SUPPORT_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
ax.plot(live_plot_equal_support_merged["bin_mid_um"], live_plot_equal_support_merged["mean_value"], color=MEASUREMENT_COLORS["fixed_tagyfp_bgz_over_dapi_gate"], lw=2.3, marker="o", ms=2.8, zorder=3)
ax.fill_between(live_plot_equal_support_merged["bin_mid_um"], live_plot_equal_support_merged["mean_value"] - live_plot_equal_support_merged["std_value"], live_plot_equal_support_merged["mean_value"] + live_plot_equal_support_merged["std_value"], color=MEASUREMENT_COLORS["fixed_tagyfp_bgz_over_dapi_gate"], alpha=0.18, zorder=2)
ax.set_title("Live FOXF1-YFP reporter activity by distance from anterior | all masked pixels merged, equal-support bins (peak-normalized)")
ax.set_ylabel("Peak-normalized live FOXF1-YFP reporter activity")
_apply_anterior_axis(ax, CENTER_TRACE_EQUAL_SUPPORT_XMIN, CENTER_TRACE_XMAX)
ax.set_ylim(*live_equal_support_merged_ylim)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_LIVE_TRACE_EQUAL_SUPPORT_MERGED_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
ax.plot(live_plot_fixedwidth_merged["bin_mid_um"], live_plot_fixedwidth_merged["weighted_mean_value"], color=MEASUREMENT_COLORS["fixed_tagyfp_bgz_over_dapi_gate"], lw=2.3, marker="o", ms=2.8, zorder=3)
ax.fill_between(live_plot_fixedwidth_merged["bin_mid_um"], live_plot_fixedwidth_merged["weighted_mean_value"] - live_plot_fixedwidth_merged["pooled_sd_value"], live_plot_fixedwidth_merged["weighted_mean_value"] + live_plot_fixedwidth_merged["pooled_sd_value"], color=MEASUREMENT_COLORS["fixed_tagyfp_bgz_over_dapi_gate"], alpha=0.18, zorder=2)
ax.set_title("Live FOXF1-YFP reporter activity by distance from anterior | all masked pixels merged, fixed-width bins (peak-normalized)")
ax.set_ylabel("Peak-normalized live FOXF1-YFP reporter activity")
_apply_anterior_axis(ax, CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)
ax.set_ylim(*live_fixedwidth_merged_ylim)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_LIVE_TRACE_FIXED_WIDTH_MERGED_PNG, dpi=180, bbox_inches="tight")
plt.show()


## Fixed Channels Vs Center Distance

In [ ]:

dapi_gate_threshold = float(fixed_global_bg_df.loc[fixed_global_bg_df["channel_key"] == "dapi", "analysis_gate_threshold"].iloc[0])
bg_mu_map = fixed_global_bg_df.set_index("channel_key")["mu_bg_raw"].astype(float).to_dict()
bg_sigma_map = fixed_global_bg_df.set_index("channel_key")["sigma_bg_raw"].astype(float).to_dict()


def _channel_bg_standardize(raw: np.ndarray, mu_bg: float, sigma_bg: float) -> np.ndarray:
    return ((np.asarray(raw, dtype=np.float32) - float(mu_bg)) / max(float(sigma_bg), float(lfq.EPS))).astype(np.float32)


fixed_rows = []
fixed_pixel_payloads_by_measurement = {meas: {} for meas in MEASUREMENT_ORDER}
fixed_analysis_payloads = {}
for pos in fixed_pos_df["canonical_position"].astype(str).tolist():
    if str(pos) not in center_df.set_index("canonical_position").index:
        continue
    payload = fixed_mask_payloads[str(pos)]
    bundle = load_fixed_bundle(pos)
    final_mask = np.asarray(payload["final_mask"], dtype=bool)
    dapi_raw = np.asarray(bundle["dapi_raw"], dtype=np.float32)
    finite_all = np.isfinite(dapi_raw) & np.isfinite(bundle["tagyfp_raw"]) & np.isfinite(bundle["sox2_raw"]) & np.isfinite(bundle["t_raw"])
    dapi_gate_keep = finite_all & (dapi_raw >= dapi_gate_threshold)
    analysis_mask = final_mask & dapi_gate_keep

    tagyfp_bgz = _channel_bg_standardize(bundle["tagyfp_raw"], bg_mu_map["tagyfp"], bg_sigma_map["tagyfp"])
    sox2_bgz = _channel_bg_standardize(bundle["sox2_raw"], bg_mu_map["sox2"], bg_sigma_map["sox2"])
    t_bgz = _channel_bg_standardize(bundle["t_raw"], bg_mu_map["t"], bg_sigma_map["t"])
    with np.errstate(divide="ignore", invalid="ignore"):
        tagyfp_ratio = np.asarray(tagyfp_bgz / np.maximum(dapi_raw, float(lfq.EPS)), dtype=np.float32)
        sox2_ratio = np.asarray(sox2_bgz / np.maximum(dapi_raw, float(lfq.EPS)), dtype=np.float32)
        t_ratio = np.asarray(t_bgz / np.maximum(dapi_raw, float(lfq.EPS)), dtype=np.float32)

    rr = center_df.set_index("canonical_position").loc[str(pos)]
    dist_um = _center_distance_map_um(
        image_shape_yx=dapi_raw.shape,
        center_x_px=float(rr["selected_fixed_x_px"]),
        center_y_px=float(rr["selected_fixed_y_px"]),
        pixel_um_x=float(bundle["image"].pixel_um_x),
        pixel_um_y=float(bundle["image"].pixel_um_y),
    )

    signal_map = {
        "fixed_tagyfp_bgz_over_dapi_gate": tagyfp_ratio,
        "fixed_sox2_bgz_over_dapi_gate": sox2_ratio,
        "fixed_t_bgz_over_dapi_gate": t_ratio,
    }
    for meas_name, arr in signal_map.items():
        stats = lfq.masked_distance_bin_stats(
            value_img=arr,
            distance_um_map=dist_um,
            mask=analysis_mask,
            bin_um=float(CENTER_BIN_UM),
        )
        for _, tr in stats.iterrows():
            fixed_rows.append(
                {
                    "canonical_position": str(pos),
                    "image_id": str(bundle["row"].get("image_id", f"{FIXED_COHORT_ID}_{pos}")),
                    "cohort_id": FIXED_COHORT_ID,
                    "measurement_name": str(meas_name),
                    "bin_idx": int(tr["bin_idx"]),
                    "bin_start_um": float(tr["bin_start_um"]),
                    "bin_end_um": float(tr["bin_end_um"]),
                    "bin_mid_um": float(tr["bin_mid_um"]),
                    "count_px": int(tr["count_px"]),
                    "mean_value": float(tr["mean_value"]),
                    "median_value": float(tr["median_value"]),
                    "std_value": float(tr["std_value"]) if pd.notna(tr["std_value"]) else np.nan,
                    "sem_value": float(tr["sem_value"]) if pd.notna(tr["sem_value"]) else np.nan,
                }
            )
        keep = analysis_mask & np.isfinite(arr) & np.isfinite(dist_um)
        fixed_pixel_payloads_by_measurement[meas_name][str(pos)] = {
            "distance_um": np.asarray(dist_um[keep], dtype=np.float32),
            "value": np.asarray(arr[keep], dtype=np.float32),
        }
    fixed_analysis_payloads[str(pos)] = {
        "bundle": bundle,
        "analysis_mask": analysis_mask,
        "tagyfp_ratio": tagyfp_ratio,
        "sox2_ratio": sox2_ratio,
        "t_ratio": t_ratio,
        "distance_um": dist_um,
    }

fixed_center_stats_df = pd.DataFrame(fixed_rows).sort_values(["measurement_name", "canonical_position", "bin_idx"]).reset_index(drop=True)
fixed_center_stats_df.to_csv(CENTER_FIXED_PIXEL_BIN_STATS_TSV, sep="	", index=False)

fixed_center_trace_df = aggregate_trace_across_images(fixed_center_stats_df)
fixed_center_trace_df.to_csv(CENTER_FIXED_TRACE_ACROSS_IMAGES_TSV, sep="	", index=False)

fixed_center_merged_df = weighted_trace_pixels(fixed_center_stats_df)
fixed_center_merged_df.to_csv(CENTER_FIXED_TRACE_ALL_PIXELS_TSV, sep="	", index=False)

if len(fixed_center_merged_df) == 0:
    fixed_equal_support_target_px = 1
else:
    shared_support = fixed_center_merged_df[fixed_center_merged_df["measurement_name"] == MEASUREMENT_ORDER[0]]["total_count_px"].astype(float)
    fixed_equal_support_target_px = int(np.nanmedian(shared_support)) if len(shared_support) else int(np.nanmedian(fixed_center_merged_df["total_count_px"].astype(float)))

equal_support_tables = []
equal_support_agg_tables = []
for meas_name in MEASUREMENT_ORDER:
    payloads = fixed_pixel_payloads_by_measurement[meas_name]
    if not payloads:
        continue
    dist_all = np.concatenate([np.asarray(v["distance_um"], dtype=np.float32) for v in payloads.values() if len(v["distance_um"])])
    val_all = np.concatenate([np.asarray(v["value"], dtype=np.float32) for v in payloads.values() if len(v["value"])])
    eq_df = equal_support_trace(dist_all, val_all, pixels_per_bin=fixed_equal_support_target_px, measurement_name=meas_name)
    equal_support_tables.append(eq_df)
    equal_support_agg_tables.append(aggregate_equal_support_across_images(payloads=payloads, equal_support_df=eq_df, measurement_name=meas_name))

fixed_center_equal_support_df = pd.concat(equal_support_tables, ignore_index=True) if equal_support_tables else pd.DataFrame()
fixed_center_equal_support_df.to_csv(CENTER_FIXED_TRACE_EQUAL_SUPPORT_TSV, sep="	", index=False)

fixed_center_equal_support_across_images_df = pd.concat(equal_support_agg_tables, ignore_index=True) if equal_support_agg_tables else pd.DataFrame()
fixed_center_equal_support_across_images_df.to_csv(CENTER_FIXED_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV, sep="	", index=False)

main_agg_raw = fixed_center_trace_df.copy().sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)
alt_agg_raw = fixed_center_equal_support_across_images_df.copy().sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)
plot_fixedwidth_per_df, plot_main_agg, _ = _normalized_trace_inputs(
    fixed_center_stats_df,
    main_agg_raw,
    mean_col="mean",
    sd_col="sd",
    measurement_order=MEASUREMENT_ORDER,
    peak_xmin=CENTER_TRACE_FIXEDWIDTH_XMIN,
    peak_xmax=CENTER_TRACE_XMAX,
)
_, plot_alt_agg, _ = _normalized_trace_inputs(
    fixed_center_stats_df,
    alt_agg_raw,
    mean_col="mean",
    sd_col="sd",
    measurement_order=MEASUREMENT_ORDER,
    peak_xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN,
    peak_xmax=CENTER_TRACE_XMAX,
)
plot_main_agg_display = plot_main_agg.copy()
plot_main_agg_display["sd"] = 1.96 * pd.to_numeric(plot_main_agg_display["sem"], errors="coerce").astype(float)
plot_alt_agg_display = plot_alt_agg.copy()
plot_alt_agg_display["sd"] = 1.96 * pd.to_numeric(plot_alt_agg_display["sem"], errors="coerce").astype(float)

plot_fixedwidth_merged = fixed_center_merged_df.copy()
for meas_name in MEASUREMENT_ORDER:
    peak = _peak_within_window(plot_fixedwidth_merged.rename(columns={"weighted_mean_value": "mean_tmp"}), mean_col="mean_tmp", xmin=CENTER_TRACE_FIXEDWIDTH_XMIN, xmax=CENTER_TRACE_XMAX, measurement_name=meas_name)
    mask = plot_fixedwidth_merged["measurement_name"] == meas_name
    for col in ["weighted_mean_value", "pooled_sd_value", "pooled_sem_value"]:
        plot_fixedwidth_merged.loc[mask, col] = pd.to_numeric(plot_fixedwidth_merged.loc[mask, col], errors="coerce").astype(float) / peak

plot_equal_support_merged = fixed_center_equal_support_df.copy()
for meas_name in MEASUREMENT_ORDER:
    peak = _peak_within_window(plot_equal_support_merged, mean_col="mean_value", xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN, xmax=CENTER_TRACE_XMAX, measurement_name=meas_name)
    mask = plot_equal_support_merged["measurement_name"] == meas_name
    for col in ["mean_value", "median_value", "std_value", "sem_value"]:
        plot_equal_support_merged.loc[mask, col] = pd.to_numeric(plot_equal_support_merged.loc[mask, col], errors="coerce").astype(float) / peak

FOXF1_MEAS = "fixed_tagyfp_bgz_over_dapi_gate"
foxf1_equal_support_ylim = _windowed_ylim(plot_alt_agg_display, mean_col="mean", band_col="sd", xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN, xmax=CENTER_TRACE_XMAX, measurement_name=FOXF1_MEAS, floor=-0.2, ceil_min=1.0)
foxf1_fixedwidth_ylim = _windowed_ylim(plot_main_agg_display, mean_col="mean", band_col="sd", xmin=CENTER_TRACE_FIXEDWIDTH_XMIN, xmax=CENTER_TRACE_XMAX, measurement_name=FOXF1_MEAS, floor=-0.2, ceil_min=1.0)

single_channel_ylim_by_meas = {}
for meas_name in ["fixed_sox2_bgz_over_dapi_gate", "fixed_t_bgz_over_dapi_gate"]:
    single_channel_ylim_by_meas[meas_name] = {
        "equal_support": _windowed_ylim(plot_alt_agg_display, mean_col="mean", band_col="sd", xmin=CENTER_TRACE_EQUAL_SUPPORT_XMIN, xmax=CENTER_TRACE_XMAX, measurement_name=meas_name, floor=0.0, ceil_min=1.0),
        "fixed_width": _windowed_ylim(plot_main_agg_display, mean_col="mean", band_col="sd", xmin=CENTER_TRACE_FIXEDWIDTH_XMIN, xmax=CENTER_TRACE_XMAX, measurement_name=meas_name, floor=0.0, ceil_min=1.0),
    }

subset_order = ["fixed_sox2_bgz_over_dapi_gate", "fixed_t_bgz_over_dapi_gate"]
subset_equal_support_ylim = (
    min(single_channel_ylim_by_meas[m]["equal_support"][0] for m in subset_order),
    max(single_channel_ylim_by_meas[m]["equal_support"][1] for m in subset_order),
)
subset_fixedwidth_ylim = (
    min(single_channel_ylim_by_meas[m]["fixed_width"][0] for m in subset_order),
    max(single_channel_ylim_by_meas[m]["fixed_width"][1] for m in subset_order),
)
fixed_merged_ylim = (-0.2, 1.2)


## Fixed Single-Channel Center-Distance Plots

In [ ]:
fig, axes = plt.subplots(len(MEASUREMENT_ORDER), 2, figsize=(14.5, 4.8 * len(MEASUREMENT_ORDER)), constrained_layout=True)
if len(MEASUREMENT_ORDER) == 1:
    axes = np.asarray([axes])
for row_idx, meas_name in enumerate(MEASUREMENT_ORDER):
    color = MEASUREMENT_COLORS[meas_name]
    label = MEASUREMENT_LABELS[meas_name]
    ax = axes[row_idx, 0]
    _plot_single_fixed_channel_trace(
        ax=ax,
        per_image_df=plot_fixedwidth_per_df,
        agg_df=plot_alt_agg_display,
        meas_name=meas_name,
        title=f"{label} by distance from anterior | equal-support bins (peak-normalized)",
        ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
        color=color,
    )
    ylim_key = "equal_support" if meas_name in single_channel_ylim_by_meas else None
    if ylim_key is not None:
        ax.set_ylim(*single_channel_ylim_by_meas[meas_name][ylim_key])
    else:
        ax.set_ylim(*foxf1_equal_support_ylim)
    _apply_anterior_axis(ax, CENTER_TRACE_EQUAL_SUPPORT_XMIN, CENTER_TRACE_XMAX)

    ax = axes[row_idx, 1]
    _plot_single_fixed_channel_trace(
        ax=ax,
        per_image_df=plot_fixedwidth_per_df,
        agg_df=plot_main_agg_display,
        meas_name=meas_name,
        title=f"{label} by distance from anterior | fixed-width bins (peak-normalized)",
        ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
        color=color,
    )
    ylim_key = "fixed_width" if meas_name in single_channel_ylim_by_meas else None
    if ylim_key is not None:
        ax.set_ylim(*single_channel_ylim_by_meas[meas_name][ylim_key])
    else:
        ax.set_ylim(*foxf1_fixedwidth_ylim)
    _apply_anterior_axis(ax, CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)
_save_plot_all_formats(fig, CENTER_FIXED_SINGLE_CHANNEL_GRID_PNG, dpi=180, bbox_inches="tight")
plt.show()


## Fixed Three-Channel Center-Distance Plots

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(8.6, 20.5), constrained_layout=True)
_plot_multi_channel_trace(
    ax=axes[0],
    per_image_df=plot_fixedwidth_per_df,
    agg_df=plot_alt_agg_display,
    measurement_order=MEASUREMENT_ORDER,
    title="Fixed FOXF1-YFP reporter (fixed), SOX2, and TBXT by distance from anterior | equal-support bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
axes[0].set_ylim(*foxf1_equal_support_ylim)
_apply_anterior_axis(axes[0], CENTER_TRACE_EQUAL_SUPPORT_XMIN, CENTER_TRACE_XMAX)

_plot_multi_channel_trace(
    ax=axes[1],
    per_image_df=plot_fixedwidth_per_df,
    agg_df=plot_main_agg_display,
    measurement_order=MEASUREMENT_ORDER,
    title="Fixed FOXF1-YFP reporter (fixed), SOX2, and TBXT by distance from anterior | fixed-width bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
axes[1].set_ylim(*foxf1_fixedwidth_ylim)
_apply_anterior_axis(axes[1], CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)

support_sub = fixed_center_merged_df[fixed_center_merged_df["measurement_name"] == MEASUREMENT_ORDER[0]].sort_values("bin_mid_um")
axes[2].bar(support_sub["bin_mid_um"], support_sub["total_count_px"], width=CENTER_BIN_UM * 0.9, color="0.70", edgecolor="0.55")
axes[2].set_title("Retained analysis-pixel support | total pixels pooled across images, fixed-width anterior-distance bins")
axes[2].set_ylabel("Pixel count")
_apply_anterior_axis(axes[2], CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)

_plot_multi_channel_merged(
    ax=axes[3],
    merged_df=plot_equal_support_merged,
    measurement_order=MEASUREMENT_ORDER,
    title="Fixed ratios by distance from anterior | all retained pixels merged, equal-support bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
    mean_col="mean_value",
    sd_col="std_value",
    count_col="count_px",
    add_ci95=False,
)
axes[3].set_ylim(*fixed_merged_ylim)
_apply_anterior_axis(axes[3], CENTER_TRACE_EQUAL_SUPPORT_XMIN, CENTER_TRACE_XMAX)

_plot_multi_channel_merged(
    ax=axes[4],
    merged_df=plot_fixedwidth_merged,
    measurement_order=MEASUREMENT_ORDER,
    title="Fixed ratios by distance from anterior | all retained pixels merged, fixed-width bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
    mean_col="weighted_mean_value",
    sd_col="pooled_sd_value",
    count_col="total_count_px",
    add_ci95=False,
)
axes[4].set_ylim(*fixed_merged_ylim)
_apply_anterior_axis(axes[4], CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)
_save_plot_all_formats(fig, CENTER_FIXED_TRACE_COMPARISON_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_trace(
    ax=ax,
    per_image_df=plot_fixedwidth_per_df,
    agg_df=plot_alt_agg_display,
    measurement_order=MEASUREMENT_ORDER,
    title="Fixed FOXF1-YFP reporter (fixed), SOX2, and TBXT by distance from anterior | equal-support bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
ax.set_ylim(*foxf1_equal_support_ylim)
_apply_anterior_axis(ax, CENTER_TRACE_EQUAL_SUPPORT_XMIN, CENTER_TRACE_XMAX)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_FIXED_TRACE_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_trace(
    ax=ax,
    per_image_df=plot_fixedwidth_per_df,
    agg_df=plot_main_agg_display,
    measurement_order=MEASUREMENT_ORDER,
    title="Fixed FOXF1-YFP reporter (fixed), SOX2, and TBXT by distance from anterior | fixed-width bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
ax.set_ylim(*foxf1_fixedwidth_ylim)
_apply_anterior_axis(ax, CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_FIXED_TRACE_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()


## Fixed SOX2 And TBXT Center-Distance Plots

In [ ]:
subset_order = ["fixed_sox2_bgz_over_dapi_gate", "fixed_t_bgz_over_dapi_gate"]
subset_fixedwidth_per_df = plot_fixedwidth_per_df[plot_fixedwidth_per_df["measurement_name"].isin(subset_order)].copy()
subset_main_agg = plot_main_agg_display[plot_main_agg_display["measurement_name"].isin(subset_order)].copy()
subset_alt_agg = plot_alt_agg_display[plot_alt_agg_display["measurement_name"].isin(subset_order)].copy()

fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_trace(
    ax=ax,
    per_image_df=subset_fixedwidth_per_df,
    agg_df=subset_alt_agg,
    measurement_order=subset_order,
    title="SOX2 and TBXT by distance from anterior | equal-support bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
ax.set_ylim(*subset_equal_support_ylim)
_apply_anterior_axis(ax, CENTER_TRACE_EQUAL_SUPPORT_XMIN, CENTER_TRACE_XMAX)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_FIXED_SOX2_T_TRACE_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_trace(
    ax=ax,
    per_image_df=subset_fixedwidth_per_df,
    agg_df=subset_main_agg,
    measurement_order=subset_order,
    title="SOX2 and TBXT by distance from anterior | fixed-width bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
ax.set_ylim(*subset_fixedwidth_ylim)
_apply_anterior_axis(ax, CENTER_TRACE_FIXEDWIDTH_XMIN, CENTER_TRACE_XMAX)
plt.tight_layout()
_save_plot_all_formats(fig, CENTER_FIXED_SOX2_T_TRACE_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()


## SOX2 Center-Distance Trough Review
These panels map the `~350 um` SOX2 trough back onto the actual fixed `SOX2 / DAPI` images used in the center-distance analysis. The cyan, red, and green annuli correspond to the neighboring fixed-width bins around the trough.

In [ ]:
sox2_fixed = fixed_center_stats_df[fixed_center_stats_df["measurement_name"] == "fixed_sox2_bgz_over_dapi_gate"].copy()
sox2_pivot = sox2_fixed.pivot_table(index="canonical_position", columns="bin_mid_um", values="mean_value")
review_bins = [297.5, 367.5, 437.5]
missing_review_bins = [v for v in review_bins if v not in sox2_pivot.columns]
if missing_review_bins:
    raise KeyError(f"Missing SOX2 review bins: {missing_review_bins}")

sox2_trough_score = ((sox2_pivot[297.5] + sox2_pivot[437.5]) / 2.0 - sox2_pivot[367.5]).sort_values(ascending=False)
sox2_review_positions = sox2_trough_score.head(6).index.astype(str).tolist()
center_review_map = center_df.set_index("canonical_position")
review_bands = [
    ("inner\n280-315 um", 280.0, 315.0, "#4dd0e1"),
    ("dip\n350-385 um", 350.0, 385.0, "#ff4d4d"),
    ("outer\n420-455 um", 420.0, 455.0, "#8bc34a"),
]

review_rows = []
fig, axes = plt.subplots(len(sox2_review_positions), 4, figsize=(14.0, 3.0 * len(sox2_review_positions)), constrained_layout=True)
if len(sox2_review_positions) == 1:
    axes = axes[None, :]

for ridx, pos in enumerate(sox2_review_positions):
    payload = fixed_analysis_payloads[str(pos)]
    sox2_ratio = np.asarray(payload["sox2_ratio"], dtype=np.float32)
    analysis_mask = np.asarray(payload["analysis_mask"], dtype=bool)
    dist_um = np.asarray(payload["distance_um"], dtype=np.float32)
    rr = center_review_map.loc[str(pos)]
    disp = common._robust_rescale(np.where(analysis_mask, sox2_ratio, np.nan))
    base = np.clip(np.dstack([disp, disp, disp]), 0.0, 1.0)

    ax = axes[ridx, 0]
    ax.imshow(base)
    ax.contour(analysis_mask.astype(float), levels=[0.5], colors="white", linewidths=0.5, alpha=0.8)
    for _, lo_um, hi_um, color in review_bands:
        band_mask = analysis_mask & (dist_um >= float(lo_um)) & (dist_um < float(hi_um))
        if np.any(band_mask):
            ax.contour(band_mask.astype(float), levels=[0.5], colors=color, linewidths=1.0)
    ax.scatter([float(rr["selected_fixed_x_px"])], [float(rr["selected_fixed_y_px"])], s=35, c="white", edgecolors="black", linewidths=0.7)
    ax.set_title(f"{pos} | SOX2 / DAPI\nall bands")
    ax.set_axis_off()

    band_means = {}
    for cidx, (label, lo_um, hi_um, color) in enumerate(review_bands, start=1):
        band_mask = analysis_mask & (dist_um >= float(lo_um)) & (dist_um < float(hi_um))
        vals = sox2_ratio[band_mask]
        vals = vals[np.isfinite(vals)]
        band_means[label] = float(np.mean(vals)) if vals.size else np.nan
        overlay = base.copy()
        if np.any(band_mask):
            rgb = np.array(mcolors.to_rgb(color), dtype=np.float32)
            overlay[band_mask] = np.clip(rgb[None, :] * 0.85 + overlay[band_mask] * 0.25, 0.0, 1.0)
        ax = axes[ridx, cidx]
        ax.imshow(overlay)
        ax.contour(analysis_mask.astype(float), levels=[0.5], colors="white", linewidths=0.4, alpha=0.6)
        ax.scatter([float(rr["selected_fixed_x_px"])], [float(rr["selected_fixed_y_px"])], s=28, c="white", edgecolors="black", linewidths=0.6)
        ax.set_title(f"{pos} | {label}\nmean={band_means[label]:.5f}")
        ax.set_axis_off()

    review_rows.append({
        "canonical_position": str(pos),
        "trough_score": float(sox2_trough_score.loc[str(pos)]),
        "inner_mean_280_315": band_means["inner\n280-315 um"],
        "dip_mean_350_385": band_means["dip\n350-385 um"],
        "outer_mean_420_455": band_means["outer\n420-455 um"],
    })

fig.suptitle(
    "SOX2 / DAPI images driving the ~350 um center-distance trough\nExact notebook-10 ratio image, selected center, and adjacent annuli",
    fontsize=14,
)
_save_plot_all_formats(fig, SOX2_CENTER_DIP_REVIEW_PNG, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

sox2_center_dip_review_df = pd.DataFrame(review_rows)
sox2_center_dip_review_df.to_csv(SOX2_CENTER_DIP_REVIEW_TSV, sep="	", index=False)
display(sox2_center_dip_review_df)
print(f"Saved {SOX2_CENTER_DIP_REVIEW_PNG}")
print(f"Saved {SOX2_CENTER_DIP_REVIEW_TSV}")
